1. Minimize the square of the difference of matrix elements between transformed camera and laser images by varying the five 
    parameters (translation in x & y, rotation about z-axis at image center, and shear in x & y) involved in the 2D affine transformation.
2. Profit $$$

In [1]:
from skimage import io,transform,util
from scipy.optimize import minimize
import numpy as np

In [2]:
#Read in tif image files.
img_stem=r'C:\Users\jglic\Downloads\3-9-26 beads'
camera_in = io.imread(img_stem+r'\YFP-camera_1952_fov3.tif')
laser_in = io.imread(img_stem+r'\eyrbow-yellowconf_fov3.tif')
camera_in=camera_in/np.max(camera_in) #normalize the matrices.
laser_in=laser_in/np.max(laser_in)

In [3]:
# Rescale camera image to new size prior to choosing coords for calculating affine transform.
# path_2048=img_stem+r'\11-7-25 2048_400nm\TRITC_2048_fov1.tif'
def resize_img(img_or_path):
    """
    Args: Camera image file path or array
    Outputs: Downscaled camera image as array
    """
    if isinstance(img_or_path, (str, bytes)):
        img_raw = io.imread(img_or_path) #file path
    else:
        img_raw = np.asarray(img_or_path) #image matrix
    img_downsc=transform.rescale(img_raw,0.25, anti_aliasing=True)
    img_out = np.zeros((512,512),dtype=float) # hard coded size, okay.
    shape0,shape1 = img_downsc.shape
    img_out[:shape0,:shape1] = img_downsc/np.max(img_downsc) #account for dimension cutoff in full camera fov. Also normalize image here.
    # io.imsave(path[0:-4]+'_downscaled.tif',util.img_as_float32(img_out)) #save edited image as new file with updated name.
    # print("Resized image saved as: "+path[0:-4]+'_downscaled.tif')
    return img_out

# test_down=resize_2048(path_2048) #test output for troubleshooting
camera_in=resize_img(camera_in)

In [ ]:
# io.imsave(img_stem+r'\RESCALED.tif',util.img_as_float(camera_in)) #save edited image as new file with updated name.

C:\Users\jglic\AppData\Local\Temp\ipykernel_20540\806730068.py:1: UserWarning: C:\Users\jglic\Downloads\3-9-26 beads\RESCALED.tif is a low contrast image
  io.imsave(img_stem+r'\RESCALED.tif',util.img_as_float(camera_in)) #save edited image as new file with updated name.


In [5]:
#Define the 2D affine transformation objective function as a function of its five parameters, 
# and have it return the .
def afftrans_obj(params: list):
    """
    Args: List of parameters for skimage.transform; translation, rotation, shear, and scaling. 
    Outputs: Sum of the squared difference in matrices between transformed source (camera) and destination (laser) detectors.
    """
    sx,sy,tx,ty,shx,shy=params[0],params[1],params[2],params[3],params[5],params[6] #unpack affine transform parameters.
    theta=params[4] #unpack transform.rotate()'s parameter.
    tform=transform.AffineTransform(scale=(sx,sy),translation=(tx,ty),shear=(shx,shy)) #generate the transform with given params.
    camera_rot=transform.rotate(camera_in,theta) #rotate the raw image.
    camera_warped=transform.warp(camera_rot,tform.inverse) #apply the affine transform to the rotated camera image.
    score=np.sum(np.square(laser_in-camera_warped)) #for each param set, assign a score based upon how well it minimizes the difference between warped input and desired output.
    return score


In [6]:
def afftrans(params: list): #Perform transformation using the optimized parameters.
    sx,sy,tx,ty,shx,shy=params[0],params[1],params[2],params[3],params[5],params[6] #unpack affine transform parameters.
    theta=params[4] #unpack transform.rotate()'s parameter.
    tform=transform.AffineTransform(scale=(sx,sy),translation=(tx,ty),shear=(shx,shy)) #generate the transform with given params.
    camera_rot=transform.rotate(camera_in,theta) #rotate the raw image.
    camera_warped=transform.warp(camera_rot,tform.inverse) #apply the affine transform to the rotated camera image.
    io.imsave(img_stem+r'\YFP-camera_2048_fov1_afftransf.tif',util.img_as_float(camera_warped)) #save edited image as new file with updated name.
    return "Sum of transformed matrix = " + str(np.sum(camera_warped))

In [7]:
#Parameter guesses and bounds for optimization function.
# g0=np.array([1.08e+00,  1.08e+00, -20, -15, 87.8, 6.487e-03, -5.678e-03])
# g0=[1.055,  1.055, -9.804, -25.15,  87.70, 6.480e-03, -5.700e-03]
g0=[ 1.056e+00,  1.055e+00, -9.700e+00, -2.550e+01,  8.770e+01, 6.000e-03, -6.451e-03]
g0bounds=[ (1.05,1.06),  (1.05,1.06), (-13,-9), (-26,-22),  (87.6,87.9), (4e-3,8e-3), (-8e-3,-5e-3)]
# g0bounds=[(None,None),(None,None),(None,None),(None,None),(None,None),(None,None),(None,None)] #bounds on parameter space.

In [8]:
soln=minimize(afftrans_obj,x0=g0,bounds=g0bounds,method='Powell',tol=1e-20) #run the chosen minimization algorithm.
soln

 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 1593.2792907386565
       x: [ 1.050e+00  1.050e+00 -1.300e+01 -2.201e+01  8.760e+01
            8.000e-03 -8.000e-03]
     nit: 7
   direc: [[-1.279e-16 -1.279e-16 ... -4.428e-16 -1.289e-16]
           [ 0.000e+00  1.000e+00 ...  0.000e+00  0.000e+00]
           ...
           [ 9.856e-12  1.331e-11 ... -3.127e-12  3.339e-12]
           [ 3.874e-17 -0.000e+00 ...  5.267e-17  2.966e-17]]
    nfev: 3533

In [9]:
afftrans(soln['x'])

C:\Users\jglic\AppData\Local\Temp\ipykernel_20540\3154511283.py:7: UserWarning: C:\Users\jglic\Downloads\3-9-26 beads\YFP-camera_2048_fov1_afftransf.tif is a low contrast image
  io.imsave(img_stem+r'\YFP-camera_2048_fov1_afftransf.tif',util.img_as_float(camera_warped)) #save edited image as new file with updated name.


'Sum of transformed matrix = 20264.87930072789'

In [ ]:
# okay yeah that worked better [ 1.065e+00  1.065e+00 -1.087e+01 -2.700e+01  8.770e+01 6.480e-03 -5.700e-03]
#[ 1.060e+00  1.060e+00 -1.000e+01 -2.500e+01  8.770e+01  6.480e-03 -5.600e-03] good as well

#Using OME-XML metadata, expect scaling to be 1.05 factor for 400 nm images (from 11-7-25). However, that appears to be too low of scaling.
###go with [ 1.060,  1.060, -9.804, -25.15,  87.70, 6.480e-03, -5.700e-03]

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 1 has 3 dimension(s)